In [31]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import ast
import re

In [32]:
print("Loading model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading model...


In [33]:
recipes_path = '../clustering/clustered_recipes.csv'
df = pd.read_csv(recipes_path)
df.head()

,Name,Ingredients_List,Ingredients_Names,Procedure,Nutrition_Facts,Calories,Carbohydrates,Protein,Fat,Saturated Fat,...,Sugar,Scaled_Calories,Scaled_Carbohydrates,Scaled_Protein,Scaled_Fat,Scaled_Saturated Fat,Scaled_Sodium,Scaled_Sugar,Cluster,Cluster_Name
0,thai-fish-cakes,"['1 pound white fish fillet , cut into 1-inch ...","['white fish fillet', 'red curry paste', 'larg...","['Fish cake:', 'Combine the fish, curry paste,...","['Serving: 1of the 6 servings', 'Calories: 308...",308.0,21.8,16.3,16.5,3.9,...,10.9,0.129310,0.140144,0.287186,0.289290,0.193493,1.198400,1.091783,3,Balanced Carb-Mains
1,budae-jjigae,"['1 tablespoon peanut oil (or vegetable oil)',...","['peanut oil', 'ground beef', 'green onions', ...",['Heat oil in a 4-quart dutch oven (or heavy p...,"['Serving: 4servings', 'Calories: 522kcal', 'C...",522.0,26.6,39.1,29.1,8.3,...,10.1,1.206753,0.364220,1.499514,1.134243,0.964090,1.701154,1.016798,1,Rich Meat Mains
2,drunken-chicken,['2 bone-in skin-on chicken leg quarters (incl...,"['bone-in skin-on chicken leg quarters', 'salt...",['Combine the salt and Sichuan peppercorns in ...,"['Serving: 1serving', 'Calories: 99kcal', 'Car...",99.0,3.0,9.9,5.1,1.4,...,2.0,-1.330232,-1.500381,-0.268062,-0.972870,-0.743717,-0.767056,-0.558707,0,Light Sides & Soups
3,matcha-tiramisu,['2 tablespoons ceremonial grade matcha and ex...,"['ceremonial grade matcha', 'very hot water', ...",['To make the matcha soak: Sift matcha into a ...,"['Serving: 1serving', 'Calories: 224kcal', 'Ca...",224.0,25.5,7.0,10.8,5.8,...,12.6,-0.379936,0.315679,-0.602039,-0.235396,0.595562,-1.274549,1.233359,2,Desserts & Sweets
4,clams-in-black-bean-sauce,"['2 lbs manila clams', 'Salt , for soaking the...","['manila clams', 'Salt', 'vegetable oil', 'Sha...",['Place the clams in a large bowl. Add 8 cups ...,"['Serving: 1serving', 'Calories: 318kcal', 'Ca...",318.0,43.1,3.9,15.0,2.9,...,17.2,0.185721,0.958079,-1.076017,0.163840,-0.096843,2.914547,1.532037,3,Balanced Carb-Mains


## Exploration

In [34]:
ingredient = df.iloc[0]['Ingredients_Names']
lis = ast.literal_eval(ingredient)
sentence = ','.join(lis).replace(',', ' ')
print(sentence)
print(" ".join(sentence.split()))

white fish fillet red curry paste large egg whites fish sauce sugar thinly sliced green beans (*Footnote 2) Vegetable oil for frying Lime wedges  Thai sweet chili sauce Persian cucumber roasted peanuts cilantro fish sauce
white fish fillet red curry paste large egg whites fish sauce sugar thinly sliced green beans (*Footnote 2) Vegetable oil for frying Lime wedges Thai sweet chili sauce Persian cucumber roasted peanuts cilantro fish sauce


## Recipe Embeddings

In [35]:
ingredients = df['Ingredients_Names']

def clean_ingredient(ingredient):
    cleaned = str(ingredient)

    
    previous = None
    while cleaned != previous:
        previous = cleaned
        cleaned = re.sub(r'\([^()]*\)', '', cleaned)

    
    cleaned = re.sub(r'[()\"\'“”‘’]', '', cleaned).replace(',', ' ')
    return " ".join(cleaned.split())

def list_to_string(ingredient):
    ingredient_list = ast.literal_eval(ingredient)
    return " ".join(clean_ingredient(item) for item in ingredient_list).lower()

ingredients = ingredients.apply(list_to_string)

sentences = df['Name'] + " " + ingredients

In [37]:
sentence_vecs = model.encode(sentences, show_progress_bar=True)

Batches: 100%|██████████| 60/60 [00:01<00:00, 35.74it/s]


In [38]:
sentence_vecs

array([[-0.08076855, -0.0052215 ,  0.00342854, ...,  0.10894916,
         0.01222055,  0.05635522],
       [-0.12167404,  0.03178843,  0.01280355, ...,  0.02288642,
         0.03169387,  0.06685385],
       [-0.08237243,  0.00894087,  0.02300057, ..., -0.03242945,
        -0.00147217,  0.06095621],
       ...,
       [-0.03142004, -0.01898925,  0.07093857, ..., -0.06971993,
         0.06101172,  0.00074974],
       [-0.11388367, -0.01620197,  0.10022826, ...,  0.07256049,
         0.04026258,  0.00412271],
       [-0.03430234, -0.04806596,  0.03992134, ...,  0.08871293,
        -0.00121886,  0.07686899]], dtype=float32)

In [39]:
sentence_vecs.shape

(1903, 384)

In [40]:
out = 'recipe_embeddings.npy'
np.save(out, sentence_vecs)

## Single Ingredient Embeddings

In [41]:
ingredients_list = df['Ingredients_Names']


ingredients_list = ingredients_list.apply(lambda x: ast.literal_eval(x)).tolist()

ingredients_list = [
    cleaned
    for sublist in ingredients_list
    for item in sublist
    if (cleaned := clean_ingredient(item))
]

print(ingredients_list[:10])

['white fish fillet', 'red curry paste', 'large egg whites', 'fish sauce', 'sugar', 'thinly sliced green beans', 'Vegetable oil for frying', 'Lime wedges', 'Thai sweet chili sauce', 'Persian cucumber']


In [44]:

ingredients_list = list(set(ingredients_list))
ingredients_vecs = model.encode(ingredients_list, show_progress_bar=True)

Batches: 100%|██████████| 120/120 [00:01<00:00, 75.06it/s]


In [43]:
individual_ingredients_file = 'individual_ingredients.csv'
df_ingredients = pd.DataFrame({'ingredient': ingredients_list})

df_ingredients.to_csv(
    individual_ingredients_file,
    index=True,
    encoding="utf-8"
)


In [45]:
out = 'ingredient_embeddings.npy'
np.save(out, ingredients_vecs)